# Análise estrutural das séries no InfluxDB

Este notebook tem como objetivo inspecionar como os datapoints estão materializados no InfluxDB, para verificar se a lógica atual de extração está correta.

## Perguntas que queremos responder

1. A tag `deviceId` contém:
   - apenas o `device_id` base?
   - ou o identificador completo do datapoint?

2. Existe alguma tag separada para:
   - `measurement_index`
   - fase
   - índice vetorial

3. Para medições como `iAcGrid` e `pAcGrid`, as fases 1, 2 e 3 estão:
   - separadas por tag?
   - embutidas no `deviceId`?
   - ou misturadas na mesma série?

## Observação
Este notebook é apenas investigativo e não altera o código do projeto.

In [7]:
import os
import pandas as pd
from dotenv import load_dotenv
from influxdb import InfluxDBClient

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 300)

In [8]:
load_dotenv()

INFLUX_HOST = os.getenv("INFLUX_HOST")
INFLUX_PORT = int(os.getenv("INFLUX_PORT", "8086"))
INFLUX_USER = os.getenv("INFLUX_USER")
INFLUX_PASSWORD = os.getenv("INFLUX_PASSWORD")
INFLUX_DATABASE = os.getenv("INFLUX_DATABASE")

client = InfluxDBClient(
    host=INFLUX_HOST,
    port=INFLUX_PORT,
    username=INFLUX_USER,
    password=INFLUX_PASSWORD,
    database=INFLUX_DATABASE,
    ssl=True,
    verify_ssl=False,
)

print("Conexão criada com sucesso.")
print(f"Database: {INFLUX_DATABASE}")

Conexão criada com sucesso.
Database: iot


In [9]:
def run_query(query: str) -> pd.DataFrame:
    """
    Executa uma query no InfluxDB e retorna um DataFrame.
    """
    result = client.query(query)
    points = list(result.get_points())
    return pd.DataFrame(points)

In [10]:
measurement_id = "iAcGrid"

In [11]:
query = f'SHOW TAG KEYS FROM "{measurement_id}"'
df_tag_keys = run_query(query)

print(f"TAG KEYS da measurement {measurement_id}")
df_tag_keys

/home/leandro_mario/workspace/dsm-influxdb-extraction/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '147.91.50.80'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


TAG KEYS da measurement iAcGrid


,tagKey
0,deviceId
1,measurementIndex
2,topic


In [12]:
query = f'SHOW FIELD KEYS FROM "{measurement_id}"'
df_field_keys = run_query(query)

print(f"FIELD KEYS da measurement {measurement_id}")
df_field_keys

/home/leandro_mario/workspace/dsm-influxdb-extraction/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '147.91.50.80'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


FIELD KEYS da measurement iAcGrid


,fieldKey,fieldType
0,value,float


In [13]:
query = f'SHOW TAG VALUES FROM "{measurement_id}" WITH KEY = "deviceId"'
df_device_values = run_query(query)

print(f"Quantidade de valores distintos de deviceId em {measurement_id}: {len(df_device_values)}")
df_device_values.head(3)

/home/leandro_mario/workspace/dsm-influxdb-extraction/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '147.91.50.80'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Quantidade de valores distintos de deviceId em iAcGrid: 86


,key,value
0,deviceId,https://react2020.eu/device/MID-0000406714D186C9
1,deviceId,https://react2020.eu/device/MID-0000406714D186C9-LEO
2,deviceId,https://react2020.eu/device/MID-00008484E913D3DB


In [14]:
device_fragment = "SMA-3008628305-EDMM"

df_device_filtered = df_device_values[
    df_device_values["value"].astype(str).str.contains(device_fragment, na=False)
].copy()

print(f"Valores de deviceId contendo '{device_fragment}': {len(df_device_filtered)}")
df_device_filtered

Valores de deviceId contendo 'SMA-3008628305-EDMM': 1


,key,value
31,deviceId,https://react2020.eu/device/SMA-3008628305-EDMM


In [15]:
query = f'SELECT * FROM "{measurement_id}" LIMIT 5'
df_sample = run_query(query)

print(f"Amostra de registros da measurement {measurement_id}")
df_sample

/home/leandro_mario/workspace/dsm-influxdb-extraction/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '147.91.50.80'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Amostra de registros da measurement iAcGrid


,time,deviceId,measurementIndex,topic,value
0,1999-12-31T23:05:00Z,https://react2020.eu/device/MID-000250E193CDD7A8-LEO,1,MID-DATAGATEWAYID001/data,0.00
1,1999-12-31T23:05:00Z,https://react2020.eu/device/MID-000269F98EED3916,1,MID-DATAGATEWAYID001/data,-11.28
2,1999-12-31T23:10:00Z,https://react2020.eu/device/MID-000250E193CDD7A8-LEO,1,MID-DATAGATEWAYID001/data,0.00
3,1999-12-31T23:10:00Z,https://react2020.eu/device/MID-000269F98EED3916,1,MID-DATAGATEWAYID001/data,-10.47
4,1999-12-31T23:15:00Z,https://react2020.eu/device/MID-000269F98EED3916,1,MID-DATAGATEWAYID001/data,-17.56


In [5]:
devices = ['https://react2020.eu/device/SMA-3008628290-EDMM','https://react2020.eu/device/SMA-3011495435-SI','https://react2020.eu/device/AI1-100000002c6d944f-OM','https://react2020.eu/device/SMA-3008628305-EDMM','https://react2020.eu/device/SMA-3012104048-SI','https://react2020.eu/device/AI3-10000000846c766e-OM','https://react2020.eu/device/SMA-3009111859-EDMM','https://react2020.eu/device/SMA-3011495437-SI','https://react2020.eu/device/AI4-100000000e70c8ee-OM']
measurements = ['iAcGrid','pAcGrid','pAcPvTot','iBat','pBat','sOc','tBat','vBat','pAcHome']
measurementIndexes = ['1','2','3']
df_final = pd.DataFrame()
for device in devices:
    for measurement_id in measurements:
          for measurementIndex in measurementIndexes:
                query = f'SELECT * FROM "{measurement_id}" WHERE "deviceId" = \'{device}\' AND "measurementIndex" = \'{measurementIndex}\' LIMIT 1'
                df = run_query(query)
                df_final = pd.concat([df_final, df], ignore_index=True)


/home/leandro_mario/workspace/dsm-influxdb-extraction/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '147.91.50.80'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/leandro_mario/workspace/dsm-influxdb-extraction/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '147.91.50.80'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/leandro_mario/workspace/dsm-influxdb-extraction/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '147.91.50.80'. Adding certificate verification is strongly advised. See: https://urllib3.read

In [ ]:
from itertools import product

devices = ['https://react2020.eu/device/SMA-3008628290-EDMM','https://react2020.eu/device/SMA-3011495435-SI','https://react2020.eu/device/AI1-100000002c6d944f-OM','https://react2020.eu/device/SMA-3008628305-EDMM','https://react2020.eu/device/SMA-3012104048-SI','https://react2020.eu/device/AI3-10000000846c766e-OM','https://react2020.eu/device/SMA-3009111859-EDMM','https://react2020.eu/device/SMA-3011495437-SI','https://react2020.eu/device/AI4-100000000e70c8ee-OM']
measurements = ['iAcGrid','pAcGrid','pAcPvTot','iBat','pBat','sOc','tBat','vBat','pAcHome']
measurementIndexes = ['1','2','3']

df_list = []

for device, measurement_id, measurement_index in product(devices, measurements, measurementIndexes):
    print(f"Processando device: {device}, measurement: {measurement_id}, measurementIndex: {measurement_index}")
    query = (
        f'SELECT * FROM "{measurement_id}" '
        f'WHERE "deviceId" = \'{device}\' '
        f'AND "measurementIndex" = \'{measurement_index}\' '
        f'LIMIT 1'
    )
    print(f"Executando query para device: {device}, measurement: {measurement_id}, measurementIndex: {measurement_index}")
    print(f"Query: {query}")
    df_list.append(run_query(query))

df_final = pd.concat(df_list, ignore_index=True)

In [6]:
df_final.to_csv("available_devices.csv", index=False)